# 04 Entry

Build the firm x product entry panel for the prediction step.

Design (no look-ahead):
- A firm-product link exists from the year of its first PQR transaction.
- For each cutoff year T, the "state" is every link that started before T. Proximity (phi)
  between products is computed only from that state.
- Risk set at T: firms active before T x products bought before T, where the firm does not
  yet supply the product.
- Outcome: entry = the firm's first transaction for that product falls in [T, T + H).
- Main predictor: density_fp = sum_{j in firm's products} phi_pj / sum_j phi_pj  (Hidalgo et al. 2007),
  the share of product p's proximity mass that the firm already covers.
Output: data/processed/entry_panel.csv


Entry models: does network proximity predict which products a firm enters next?

1. Explanatory logits on the pooled panel (cutoffs 2011-2023), SE clustered by firm.
2. Out-of-sample prediction: train on cutoffs 2011-2017, test on 2020 and 2023.
   Compare a popularity baseline, a controls-only logit, logit + density, and a random forest.
Sample: firms with a PQR transaction in the 3 years before the cutoff x products bought in
those 3 years, excluding pairs the firm already supplies.


Score potential new suppliers for thinly supplied products (policy application).

- Fit the full entry logit (density + same area + firm breadth + product popularity + cutoff FE)
  on all cutoffs 2011-2023.
- Build the current state (all links through 2025, cutoff 2026) and score every
  recently active firm x recently bought product the firm does not yet supply.
- Thin products: bought in 2021-2025 with <= 2 manufacturers supplying them in 2021-2025.
- Flag whether the candidate already holds a WHO PQ listing whose INN text contains every
  molecule in the product name (a rough signal of regulatory readiness).
Output: outputs/tables/t12_candidate_suppliers.csv


f8: entry rate by network density; f9: out-of-sample recall by model.

In [ ]:
%run 00_utils.ipynb

import re
import numpy as np, pandas as pd, statsmodels.formula.api as smf, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

set_style()

## Panel

In [ ]:
H = 3
CUTOFFS = [2011, 2014, 2017, 2020, 2023]
pqr = pd.read_csv(ROOT / "data" / "processed" / "pqr_medicines_clean.csv")
pqr = pqr[pqr.year.between(2005, 2025)]
links = pqr.groupby(["firm", "product"]).agg(first_year=("year", "min"), n_tx=("pk", "size")).reset_index()
area = pqr.groupby("product")["area"].first()
hq = pqr.groupby("firm")["hq_country"].first()

def panel_at(T):
    state = links[links.first_year < T]
    pe = project_products(state)
    phi = pd.concat([pe.rename(columns={"source": "p", "target": "q"}),
                     pe.rename(columns={"target": "p", "source": "q"})])[["p", "q", "phi"]]
    tot = phi.groupby("p")["phi"].sum()
    firms, prods = state.firm.unique(), state["product"].unique()
    have = set(zip(state.firm, state["product"]))
    grid = pd.MultiIndex.from_product([firms, prods], names=["firm", "product"]).to_frame(index=False)
    grid = grid[[(f, p) not in have for f, p in zip(grid.firm, grid["product"])]]
    # density: proximity from candidate product p to products the firm already supplies
    fp = state[["firm", "product"]].rename(columns={"product": "q"})
    cov = fp.merge(phi, on="q").groupby(["firm", "p"])["phi"].sum().rename("covered").reset_index()
    g = grid.merge(cov, left_on=["firm", "product"], right_on=["firm", "p"], how="left").drop(columns="p")
    g["covered"] = g["covered"].fillna(0)
    g["density"] = g["covered"] / g["product"].map(tot).replace(0, np.nan)
    g["density"] = g["density"].fillna(0)
    # controls
    g["firm_n_products"] = g.firm.map(state.groupby("firm")["product"].nunique())
    g["prod_n_suppliers"] = g["product"].map(state.groupby("product")["firm"].nunique())
    g["area"] = g["product"].map(area)
    fa = state.assign(area=state["product"].map(area)).groupby("firm")["area"].agg(set)
    g["same_area"] = [a in fa[f] for f, a in zip(g.firm, g.area)]
    recent = pqr[pqr.year.between(T - 3, T - 1)]
    g["firm_active_recent"] = g.firm.isin(recent.firm)
    g["prod_bought_recent"] = g["product"].isin(recent["product"])
    g["hq_country"] = g.firm.map(hq)
    fut = links[(links.first_year >= T) & (links.first_year < T + H)]
    g["entry"] = [(f, p) in set(zip(fut.firm, fut["product"])) for f, p in zip(g.firm, g["product"])]
    g["cutoff"] = T
    return g

panel = pd.concat([panel_at(T) for T in CUTOFFS], ignore_index=True)
panel.to_csv(ROOT / "data" / "processed" / "entry_panel.csv", index=False)
s = panel.groupby("cutoff").agg(rows=("entry", "size"), entries=("entry", "sum"),
                                firms=("firm", "nunique"), products=("product", "nunique"),
                                mean_density=("density", "mean"))
s["entry_rate_%"] = (100 * s.entries / s.rows).round(2)
print(s.round(3).to_string())
a = panel[panel.firm_active_recent & panel.prod_bought_recent]
print("\nrestricted to recently active firms & recently bought products:", len(a), "rows,", a.entry.sum(), "entries")
print("density: entrants vs non-entrants (median):", a.groupby("entry").density.median().round(3).to_dict())
print("entries in same disease area:", round(a[a.entry].same_area.mean(), 2), "| base rate same_area:", round(a.same_area.mean(), 2))

I used 5 cutoffs and only built proximity from links that started before each cutoff, so the predictor did not use future information. After I limited the sample to recently active firms and recently bought products, the panel had 24,111 rows and 142 entries (0.6%).

Entrants had much higher density than non-entrants (median 0.27 vs 0.01). 92% of entries were in a disease area the firm already served, compared with a base rate of 44%. Products first bought after a cutoff are outside the risk set, so entry into brand-new products is not captured.

## Models

In [ ]:
TAB = ROOT / "outputs" / "tables"
p = pd.read_csv(ROOT / "data" / "processed" / "entry_panel.csv")
p = p[p.firm_active_recent & p.prod_bought_recent].copy()
p["entry"] = p.entry.astype(int); p["same_area"] = p.same_area.astype(int)
p["log_firm_products"] = np.log(p.firm_n_products)
p["log_prod_suppliers"] = np.log(p.prod_n_suppliers)
p["density_z"] = (p.density - p.density.mean()) / p.density.std()
p["cutoff_f"] = p.cutoff.astype(str)

In [ ]:
# ---- 1. explanatory logits
forms = {"M1 controls": "entry ~ log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M2 + same area": "entry ~ same_area + log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M3 + density": "entry ~ density_z + log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M4 full": "entry ~ density_z + same_area + log_firm_products + log_prod_suppliers + C(cutoff_f)"}
rows = []
for name, f in forms.items():
    m = smf.logit(f, data=p).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(p.firm)[0]})
    for v in ["density_z", "same_area", "log_firm_products", "log_prod_suppliers"]:
        if v in m.params:
            rows.append({"model": name, "term": v, "odds_ratio": np.exp(m.params[v]),
                         "ci_low": np.exp(m.conf_int().loc[v, 0]), "ci_high": np.exp(m.conf_int().loc[v, 1]),
                         "p": m.pvalues[v]})
    rows.append({"model": name, "term": "pseudo_R2", "odds_ratio": m.prsquared})
    rows.append({"model": name, "term": "N / entries", "odds_ratio": len(p), "ci_low": p.entry.sum()})
coef = pd.DataFrame(rows); coef.round(4).to_csv(TAB / "t10_entry_logit.csv", index=False)
print(coef.round(3).to_string(index=False))

Density mattered beyond disease area. In the full model, one SD of density raised the odds of entry by 37% (OR 1.37, 95% CI 1.24-1.52), holding constant same area (OR 8.1), firm breadth (OR 2.1) and product popularity (OR 2.2). Pseudo R2 rose from 0.22 with same area and controls to 0.24 with density. Same area had the larger effect, and density added a smaller but clear signal.

In [ ]:
# ---- 2. out-of-sample
X_ctrl = ["log_firm_products", "log_prod_suppliers", "same_area"]
X_full = X_ctrl + ["density"]
tr, te = p[p.cutoff <= 2017], p[p.cutoff >= 2020]
def topk(y, s, frac):
    k = max(1, int(round(frac * len(y)))); idx = np.argsort(-s)[:k]
    return y[idx].sum() / y.sum()                       # recall of entries within top-k%
res, scores = [], {}
models = {
    "Popularity (n suppliers)": (["log_prod_suppliers"], LogisticRegression(max_iter=1000)),
    "Controls logit": (X_ctrl, LogisticRegression(max_iter=1000)),
    "Density logit": (X_full, LogisticRegression(max_iter=1000)),
    "Random forest": (X_full, RandomForestClassifier(n_estimators=500, min_samples_leaf=20,
                                                     class_weight="balanced", random_state=5205, n_jobs=-1)),
}
for name, (X, mdl) in models.items():
    mdl.fit(tr[X], tr.entry); s = mdl.predict_proba(te[X])[:, 1]; scores[name] = s
    y = te.entry.to_numpy()
    res.append({"model": name, "AUC": roc_auc_score(y, s), "avg_precision": average_precision_score(y, s),
                "recall_top1%": topk(y, s, .01), "recall_top5%": topk(y, s, .05), "recall_top10%": topk(y, s, .10)})
res = pd.DataFrame(res); res["base_rate"] = te.entry.mean()
res.round(3).to_csv(TAB / "t11_entry_prediction.csv", index=False)
print(f"\ntrain {len(tr)} rows / {tr.entry.sum()} entries | test {len(te)} rows / {te.entry.sum()} entries")
print(res.round(3).to_string(index=False))

# entry rate by density decile (for figure)
p["density_bin"] = pd.cut(p.density, [-0.001, 0, .05, .1, .2, .3, .5, 1.0],
                          labels=["0", "0-0.05", "0.05-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.5", "0.5+"])
d = p.groupby("density_bin", observed=True).agg(pairs=("entry", "size"), entries=("entry", "sum"))
d["entry_rate_%"] = 100 * d.entries / d.pairs
d.round(2).to_csv(TAB / "t11b_entry_by_density.csv"); print("\n", d.round(2).to_string())

I trained on the 2011-2017 cutoffs and tested on 2020 and 2023, which had 34 entries (base rate 0.5%). Popularity alone was close to random (AUC 0.55). The controls logit reached AUC 0.87, and adding density raised it to 0.89. Density also found more entries among the top predictions: 47% vs 35% in the top 5%, and 62% vs 56% in the top 10%. The random forest had a lower AUC (0.83) and lower top-5% and top-10% recall than the density logit, so I kept the simpler logit. The gain from density is modest, and 34 test entries is a small sample.

Entry rose with density across the bins, from 0.15% at zero density to 7.0% at 0.3-0.5 and 8.1% above 0.5.

## Candidate suppliers

In [ ]:
p = pd.read_csv(ROOT / "data" / "processed" / "entry_panel.csv")
p = p[p.firm_active_recent & p.prod_bought_recent].copy()
def prep(d):
    d = d.copy(); d["same_area"] = d.same_area.astype(int)
    d["log_firm_products"] = np.log(d.firm_n_products); d["log_prod_suppliers"] = np.log(d.prod_n_suppliers)
    return d
p = prep(p); p["entry"] = p.entry.astype(int)
m = smf.logit("entry ~ density + same_area + log_firm_products + log_prod_suppliers", data=p).fit(disp=0)

now = prep(panel_at(2026))
now = now[now.firm_active_recent & now.prod_bought_recent].copy()
now["p_entry_3y"] = m.predict(now)

pqr = pd.read_csv(ROOT / "data" / "processed" / "pqr_medicines_clean.csv")
rec = pqr[pqr.year.between(2021, 2025)]
cur = rec.groupby("product").agg(current_suppliers=("firm", lambda s: "; ".join(sorted(s.unique()))),
                                 n_current=("firm", "nunique"), spend_2021_25_usd_m=("cost_usd", lambda x: x.sum() / 1e6))
thin = cur[cur.n_current <= 2]

fpp = pd.read_csv(ROOT / "data" / "processed" / "pq_fpp_clean.csv")
pq_text = fpp.groupby("firm")["INN, Dosage Form and Strength"].apply(lambda s: " | ".join(s).lower())
STOP = {"tablet", "tablets", "coblister", "blister", "patient", "dispersible", "pediatric", "paediatric",
        "india", "category", "granules", "injection", "sulfate", "phosphate", "tetraphosphate"}
def molecules(prod):
    toks = re.findall(r"[a-z]{5,}", re.sub(r"\(.*?\)", " ", prod.lower()))
    return [t for t in toks if t not in STOP]
def pq_ready(firm, prod):
    mol = molecules(prod); txt = pq_text.get(firm, "")
    return bool(mol) and all(t in txt for t in mol)

c = now[now["product"].isin(thin.index)].copy()
c["who_pq_listed"] = [pq_ready(f, q) for f, q in zip(c.firm, c["product"])]
c = c.sort_values(["product", "p_entry_3y"], ascending=[True, False]).groupby("product").head(5)
c = c.merge(thin, left_on="product", right_index=True)
c["rank"] = c.groupby("product").cumcount() + 1
out = c[["area", "product", "n_current", "current_suppliers", "spend_2021_25_usd_m", "rank", "firm", "hq_country",
         "p_entry_3y", "density", "same_area", "who_pq_listed"]].sort_values(["spend_2021_25_usd_m", "rank"], ascending=[False, True])
out.round(4).to_csv(TAB / "t12_candidate_suppliers.csv", index=False)

print("thin products (<=2 suppliers in 2021-25):", len(thin), "| with scored candidates:", out["product"].nunique())
pd.set_option("display.width", 250, "display.max_colwidth", 45)
show = out[out["rank"] <= 3].groupby("product", sort=False).head(3)
top_prod = out.drop_duplicates("product").head(8)["product"]
print(show[show["product"].isin(top_prod)][["product", "current_suppliers", "spend_2021_25_usd_m", "rank", "firm",
                                            "p_entry_3y", "density", "who_pq_listed"]].round(3).to_string(index=False))
print("\ncandidates with WHO PQ listing for the molecule:", out.who_pq_listed.sum(), "of", len(out))

39 products bought in 2021-2025 had at most 2 manufacturers in that period. For 31 of them, I listed the 5 active firms with the highest predicted 3-year entry probability. Some examples:

- Bedaquiline (Janssen, Lupin): Macleods ranked first (p = 0.10) and has a WHO PQ listing
- Abacavir + dolutegravir + lamivudine (Aurobindo, Laurus): Cipla, Mylan and Macleods (p = 0.20-0.26), all with WHO PQ listings
- DHA-piperaquine (Alfasigma, Guilin): Cipla, Macleods and Ipca
- Pyronaridine-artesunate (Shin Poong only): density was 0 for every candidate, so the network gives no signal

Products like pyronaridine-artesunate are where market shaping could not rely on the network to point to likely entrants. The WHO PQ flag is a text match on molecule names and should be checked by hand.

## Figures

In [ ]:
T, F = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures"
INK, INK2, MUTED, GRID, BLUE, GRAY = "#0b0b0b", "#595959", "#8a8984", "#e6e5e1", "#003D78", "#B4B5B8"

d = pd.read_csv(T / "t11b_entry_by_density.csv")
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(d.density_bin, d["entry_rate_%"], color=BLUE, width=0.72, edgecolor="white", linewidth=1)
for i, r in d.iterrows():
    ax.text(i, r["entry_rate_%"] + 0.15, f"{r['entry_rate_%']:.1f}%", ha="center", fontsize=8.5, color=INK2)
    ax.text(i, -0.75, f"n={int(r.pairs):,}", ha="center", fontsize=7.5, color=MUTED)
ax.set_ylim(-1.1, d["entry_rate_%"].max() * 1.18); ax.axhline(0, color=GRID, lw=0.8)
ax.set_xlabel("Density: how close the product is to what the firm already supplies", labelpad=14)
ax.set_ylabel("Firms entering within 3 years (%)"); ax.grid(axis="x", visible=False)
ax.set_title("Firms enter products close to what they already make", loc="left", fontsize=12, fontweight="bold", pad=22)
ax.text(0, 1.02, "Share of firm-product pairs where the firm starts supplying the product in the next 3 years",
        transform=ax.transAxes, fontsize=9, color=INK2, va="bottom")
fig.text(0.01, -0.13, "Pooled over cutoffs 2011, 2014, 2017, 2020, 2023 (24,111 pairs, 142 entries). Density = share of the product's "
         "proximity links that the firm already covers.\nSource: Global Fund PQR.", fontsize=7.5, color=MUTED)
fig.savefig(F / "f8_entry_by_density.png"); plt.show()

r = pd.read_csv(T / "t11_entry_prediction.csv")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
y = r.model[::-1]; v = r["recall_top10%"][::-1] * 100
cols = [BLUE if m == "Density logit" else GRAY for m in y]
ax.barh(y, v, color=cols, height=0.6, edgecolor="white", linewidth=1)
for i, (val, auc) in enumerate(zip(v, r.AUC[::-1])):
    ax.text(val + 1, i, f"{val:.0f}%   (AUC {auc:.2f})", va="center", fontsize=8.5, color=INK2)
ax.set_xlim(0, 100); ax.set_xlabel("Share of actual entries found in the top 10% of predictions (%)")
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
ax.set_title("Adding network density modestly improves prediction", loc="left", fontsize=12, fontweight="bold", pad=10)
fig.text(0.01, -0.08, f"Trained on cutoffs 2011-2017, tested on 2020 and 2023 (6,868 pairs, 34 entries; base rate {r.base_rate[0]:.1%}). "
         "Controls = firm breadth, product popularity, same disease area.", fontsize=7.5, color=MUTED)
fig.savefig(F / "f9_entry_prediction.png"); plt.show()
print("ok")

There were only 142 entries, so the estimates are imprecise, and outcomes for the 2023 cutoff are cut short by reporting lags. PQR also only shows Global Fund-financed purchases, so a firm may have supplied a product elsewhere before it appears here as an entrant.